
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Build the Model (RAG Chain)

This notebook should be run before the module to create and register a model that will be used for the rest of the module.

**🚨 Note**: This notebook is run by `Workspace Setup`. You don't need to manually run this notebook.

### Classroom Setup

In [0]:
%pip install -qqq -U databricks-vectorsearch llama-index PyPDF2 databricks-sdk langchain==0.3.7 databricks-langchain mlflow-skinny[databricks]==3.4.0

dbutils.library.restartPython()

In [0]:
dbutils.library.restartPython()

In [0]:
%run ../Includes/Classroom-Setup-00

## Create Shared Catalog

In [0]:
catalog_name = "genai_shared_catalog_03"
schema_name = f"ws_{spark.conf.get('spark.databricks.clusterUsageTags.clusterOwnerOrgId')}"

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS `{catalog_name}`;")
spark.sql(f"USE CATALOG `{catalog_name}`;")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{schema_name}`;")
spark.sql(f"USE SCHEMA `{schema_name}`;")
spark.sql(f"GRANT USE CATALOG, USE SCHEMA, EXECUTE ON CATALOG `{catalog_name}` TO `account users`;")

## Helpers

In [0]:
import time
import re
import io
import os
import pandas as pd 

from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import Document
from llama_index.core.utils import set_global_tokenizer
from transformers import AutoTokenizer
from typing import Iterator
from pyspark.sql.functions import col, udf, length, pandas_udf, explode
from PyPDF2 import PdfReader

In [0]:
def parse_bytes_pypdf(raw_doc_contents_bytes: bytes):
    try:
        pdf = io.BytesIO(raw_doc_contents_bytes)
        reader = PdfReader(pdf)
        parsed_content = [page_content.extract_text() for page_content in reader.pages]
        return "\n".join(parsed_content)
    except Exception as e:
        warnings.warn(f"Exception {e} has been thrown during parsing")
        return None 
      
def extract_doc_text(x : bytes) -> str:
  # Read files and extract the values with unstructured
  sections = partition(file=io.BytesIO(x))
  def clean_section(txt):
    txt = re.sub(r'\n', '', txt)
    return re.sub(r' ?\.', '.', txt)
  # Default split is by section of document
  # concatenate them all together because we want to split by sentence instead.
  return "\n".join([clean_section(s.text) for s in sections]) 


def pprint(obj):
  import pprint
  pprint.pprint(obj, compact=True, indent=1, width=100)

def index_exists(vsc, endpoint_name, index_full_name):
  try:
      dict_vsindex = vsc.get_index(endpoint_name, index_full_name).describe()
      return dict_vsindex.get('status').get('ready', False)
  except Exception as e:
      if 'RESOURCE_DOES_NOT_EXIST' not in str(e):
          print(f'Unexpected error describing the index. This could be a permission issue.')
          raise e
  return False

def wait_for_vs_endpoint_to_be_ready(vsc, vs_endpoint_name):
  for i in range(180):
    endpoint = vsc.get_endpoint(vs_endpoint_name)
    status = endpoint.get("endpoint_status", endpoint.get("status"))["state"].upper()
    if "ONLINE" in status:
      return endpoint
    elif "PROVISIONING" in status or i <6:
      if i % 20 == 0: 
        print(f"Waiting for endpoint to be ready, this can take a few min... {endpoint}")
      time.sleep(10)
    else:
      raise Exception(f'''Error with the endpoint {vs_endpoint_name}. - this shouldn't happen: {endpoint}.\n Please delete it and re-run the previous cell: vsc.delete_endpoint("{vs_endpoint_name}")''')
  raise Exception(f"Timeout, your endpoint isn't ready yet: {vsc.get_endpoint(vs_endpoint_name)}")

def wait_for_index_to_be_ready(vsc, vs_endpoint_name, index_name):
  for i in range(180):
    idx = vsc.get_index(vs_endpoint_name, index_name).describe()
    index_status = idx.get('status', idx.get('index_status', {}))
    status = index_status.get('detailed_state', index_status.get('status', 'UNKNOWN')).upper()
    url = index_status.get('index_url', index_status.get('url', 'UNKNOWN'))
    if "ONLINE" in status:
      return
    if "UNKNOWN" in status:
      print(f"Can't get the status - will assume index is ready {idx} - url: {url}")
      return
    elif "PROVISIONING" in status:
      if i % 40 == 0: print(f"Waiting for index to be ready, this can take a few min... {index_status} - pipeline url:{url}")
      time.sleep(10)
    else:
        raise Exception(f'''Error with the index - this shouldn't happen. DLT pipeline might have been killed.\n Please delete it and re-run the previous cell: vsc.delete_index("{index_name}, {vs_endpoint_name}") \nIndex details: {idx}''')
  raise Exception(f"Timeout, your index isn't ready yet: {vsc.get_index(index_name, vs_endpoint_name)}")

## Prepare Data

1. Read raw `pdf` into `pdf_raw_text` delta table
2. Chunk using llama-index & pandas_udf
3. Materialize chunked documents into `pdf_text_chunks` delta table

In [0]:
# Reduce the arrow batch size as our PDF can be big in memory
spark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", 10)

articles_path = f"{DA.paths.datasets.arxiv}/arxiv-articles/"
table_name = f"{catalog_name}.{schema_name}.pdf_raw_text"

# read pdf files
df = (
        spark.read.format("binaryfile")
        .option("recursiveFileLookup", "true")
        .load(articles_path)
        )

# save list of the files to table
df.write.mode("overwrite").saveAsTable(table_name)

In [0]:
@pandas_udf("array<string>")
def read_as_chunk(batch_iter: Iterator[pd.Series]) -> Iterator[pd.Series]:
    #set llama2 as tokenizer
    set_global_tokenizer(
      AutoTokenizer.from_pretrained("hf-internal-testing/llama-tokenizer")
    )
    #Sentence splitter from llama_index to split on sentences
    splitter = SentenceSplitter(chunk_size=500, chunk_overlap=50)
    def extract_and_split(b):
      txt = parse_bytes_pypdf(b) 
      nodes = splitter.get_nodes_from_documents([Document(text=txt)])
      return [n.text for n in nodes]

    for x in batch_iter:
        yield x.apply(extract_and_split)

In [0]:
df_chunks = (df
                .withColumn("content", explode(read_as_chunk("content")))
                .selectExpr('path as pdf_name', 'content')
                )

In [0]:
%sql
CREATE TABLE IF NOT EXISTS pdf_text_chunks (
  id BIGINT GENERATED BY DEFAULT AS IDENTITY,
  pdf_name STRING,
  content STRING,
  embedding ARRAY <FLOAT>
  ) TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
chunks_table_name = f"{catalog_name}.{schema_name}.pdf_text_chunks"
df_chunks.write.mode("append").saveAsTable(chunks_table_name)

## Create Databricks Vector Search Endpoint

1. Spin-up VS endpoint
2. Create index from `pdf_text_chunks` delta table

In [0]:
# Grant CREATE TABLE permission needed for Vector Search index creation
current_user = spark.sql("SELECT current_user() as user").collect()[0]['user']
print(f"Current user: {current_user}")

# Grant necessary permissions
spark.sql(f"GRANT CREATE TABLE ON SCHEMA {catalog_name}.{schema_name} TO `{current_user}`")
spark.sql(f"GRANT USE SCHEMA ON SCHEMA {catalog_name}.{schema_name} TO `{current_user}`")

print(f"✓ Granted CREATE TABLE and USE SCHEMA on {catalog_name}.{schema_name}")

In [0]:
from databricks.vector_search.client import VectorSearchClient
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.vectorsearch import EndpointType
import time

vs_endpoint_name = "genai_vs_endpoint"
source_table_fullname = f"{catalog_name}.{schema_name}.pdf_text_chunks"
vs_index_fullname = f"{catalog_name}.{schema_name}.pdf_text_vs_index"

vsc = VectorSearchClient(disable_notice=True)
w = WorkspaceClient()

# -------------------------
# Utils
# -------------------------
def wait_for_vs_endpoint_to_be_ready(vsc, w, endpoint_name, timeout=900):
    """Wait for endpoint using both VectorSearchClient and SDK for reliability"""
    start = time.time()
    while time.time() - start < timeout:
        try:
            # Try SDK first (more reliable)
            endpoint = w.vector_search_endpoints.get_endpoint(endpoint_name=endpoint_name)
            if endpoint.endpoint_status and endpoint.endpoint_status.state:
                state = str(endpoint.endpoint_status.state)
                if "ONLINE" in state:
                    print(f"Endpoint is {state}")
                    return True
                elif "PROVISIONING" in state or "PENDING" in state:
                    print(f"Endpoint provisioning... (state: {state})")
                else:
                    print(f"Endpoint state: {state}")
        except Exception as e:
            if "not found" not in str(e).lower():
                print(f"Error checking endpoint: {e}")
        
        time.sleep(10)
    raise TimeoutError(f"VS Endpoint {endpoint_name} not ready after {timeout}s")

def wait_for_index_to_be_ready(vsc, endpoint_name, index_name, timeout=1800):
    """Wait for index to be ready"""
    start = time.time()
    while time.time() - start < timeout:
        try:
            # Get the index and describe it
            idx = vsc.get_index(endpoint_name, index_name).describe()
            
            # Handle both 'status' and 'index_status' keys with fallback
            index_status = idx.get('status', idx.get('index_status', {}))
            status = index_status.get('detailed_state', index_status.get('status', 'UNKNOWN')).upper()
            
            print(f"Index state: {status}")
            
            # Check if index is online/ready
            if status.startswith("ONLINE"):
                return True
        except Exception as e:
            error_str = str(e)
            if "does not exist" in error_str:
                print("Index not created yet, waiting...")
            else:
                print(f"Checking index: {error_str}")
        
        time.sleep(15)
    raise TimeoutError(f"VS Index {index_name} not ready after {timeout}s")

def index_exists(vsc, endpoint_name, index_fullname):
    """Check if index exists"""
    try:
        idx = vsc.get_index(endpoint_name, index_fullname)
        return True
    except Exception as e:
        if "does not exist" in str(e).lower() or "not found" in str(e).lower():
            return False
        else:
            raise e

def endpoint_exists(w, endpoint_name):
    """Check if endpoint truly exists using SDK"""
    try:
        endpoint = w.vector_search_endpoints.get_endpoint(endpoint_name=endpoint_name)
        return True
    except Exception as e:
        if "not found" in str(e).lower():
            return False
        raise e

# -------------------------
# Endpoint
# -------------------------
print("=" * 60)
print("STEP 1: Vector Search Endpoint")
print("=" * 60)

if not endpoint_exists(w, vs_endpoint_name):
    print(f"Creating VS endpoint: {vs_endpoint_name}...")
    w.vector_search_endpoints.create_endpoint(
        name=vs_endpoint_name,
        endpoint_type=EndpointType.STANDARD
    )
    print("Endpoint creation initiated. This may take 5-10 minutes...")
    wait_for_vs_endpoint_to_be_ready(vsc, w, vs_endpoint_name)
    print(f"✓ Endpoint {vs_endpoint_name} is ready.")
else:
    print(f"Endpoint {vs_endpoint_name} already exists.")
    endpoint = w.vector_search_endpoints.get_endpoint(endpoint_name=vs_endpoint_name)
    state = str(endpoint.endpoint_status.state) if endpoint.endpoint_status else "Unknown"
    
    if "ONLINE" in state:
        print(f"✓ Endpoint is {state}")
    else:
        print(f"Endpoint state: {state}")
        print("Waiting for endpoint to be ready...")
        wait_for_vs_endpoint_to_be_ready(vsc, w, vs_endpoint_name)
        print("✓ Endpoint is now ready.")

# -------------------------
# Index
# -------------------------
print("\n" + "=" * 60)
print("STEP 2: Vector Search Index")
print("=" * 60)

try:
    if not index_exists(vsc, vs_endpoint_name, vs_index_fullname):
        print(f"Creating index: {vs_index_fullname}")
        print(f"Source table: {source_table_fullname}")
        print(f"Embedding model: databricks-gte-large-en")
        
        vsc.create_delta_sync_index(
            endpoint_name=vs_endpoint_name,
            index_name=vs_index_fullname,
            source_table_name=source_table_fullname,
            pipeline_type="TRIGGERED",
            primary_key="id",
            embedding_dimension=1024,
            embedding_source_column="content",
            embedding_model_endpoint_name="databricks-gte-large-en"
        )
        print("\nIndex creation initiated. This may take 10-20 minutes...")
        wait_for_index_to_be_ready(vsc, vs_endpoint_name, vs_index_fullname)
        print("✓ Vector Search index is ready!")
    else:
        print(f"Index {vs_index_fullname} already exists.")
        print("Syncing index with source table...")
        vsc.get_index(vs_endpoint_name, vs_index_fullname).sync()
        print("✓ Index synced successfully!")
        
except Exception as e:
    error_msg = str(e)
    if "does not have CREATE TABLE" in error_msg:
        print("\n" + "!" * 60)
        print("PERMISSION ERROR")
        print("!" * 60)
        print(f"\nYou need CREATE TABLE permission on schema: {catalog_name}.{schema_name}")
        print("\nPlease run the cell above this one to grant permissions, or contact your admin.")
        print("\nRequired permissions:")
        print(f"  - CREATE TABLE ON SCHEMA {catalog_name}.{schema_name}")
        print(f"  - USE SCHEMA ON SCHEMA {catalog_name}.{schema_name}")
    raise

print("\n" + "=" * 60)
print("✓ VECTOR SEARCH SETUP COMPLETE")
print("=" * 60)

### Grant `SELECT` access to both catalog and created index to current users & `Account Users`
For demo purposes only

In [0]:
spark.sql(f"GRANT `USE SCHEMA` ON SCHEMA {catalog_name}.{schema_name} TO `account users`;")
spark.sql(f"GRANT SELECT ON TABLE {vs_index_fullname} TO `account users`;")

# this is required for deploying the Review App
spark.sql(f"GRANT `CREATE MODEL` ON SCHEMA {catalog_name}.{schema_name} TO `account users`;")
spark.sql(f"GRANT `CREATE TABLE` ON SCHEMA {catalog_name}.{schema_name} TO `account users`;")

## Create the Chain


In [0]:
import mlflow
from operator import itemgetter
from databricks.vector_search.client import VectorSearchClient
from databricks_langchain import ChatDatabricks, DatabricksVectorSearch, DatabricksEmbeddings
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
)
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Disable MLflow Tracing
mlflow.tracing.disable()
mlflow.langchain.autolog(disable=True, exclusive=True, log_traces=False)

vsc = VectorSearchClient(disable_notice=True)
vs_index = vsc.get_index(
    endpoint_name=vs_endpoint_name,
    index_name=vs_index_fullname
)

def vector_search_as_retriever(persist_dir=None):
    vectorstore = DatabricksVectorSearch(
        vs_index_fullname,
        columns=["id", "content", "pdf_name"]
    )
    return vectorstore.as_retriever(search_kwargs={"k": 3})

# Return the string contents of the most recent messages: [{...}] from the user to be used as input question
def extract_user_query_string(chat_messages_array):
    return chat_messages_array[-1]["content"]

def format_context(docs):
    chunk_contents = [f"Passage: {d.page_content}\n" for d in docs]
    return "".join(chunk_contents)

# Define template for prompt
prompt = ChatPromptTemplate.from_messages(
    [
        ( 
            "system",
             """You are an assistant for GENAI teaching class. You are answering questions related to Generative AI and how it impacts humans life. If the question is not related to one of these topics, kindly decline to answer. If you don't know the answer, just say that you don't know, don't try to make up an answer. Keep the answer as concise as possible. Use the following pieces of context to answer the question at the end: <context>{context}</context>"""
        ),
        # User's question
        ("user", "{question}"),
    ]
)

# Define foundation model for generating responses
model = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", max_tokens = 300)

# RAG Chain
chain = (
    {
        "question": itemgetter("messages") | RunnableLambda(extract_user_query_string),
        "context": itemgetter("messages")
        | RunnableLambda(extract_user_query_string)
        | vector_search_as_retriever
        | RunnableLambda(format_context),
    }
    | prompt
    | model
    | StrOutputParser()
)

# Let's give it a try:
input_example = {"messages": [ {"role": "user", "content": "What is Retrieval-augmented Generation?"}]}
answer = chain.invoke(input_example)
print(answer)

## Register the Model

In [0]:
import mlflow
import langchain
import langchain_community
from mlflow.models import infer_signature
import databricks.vector_search
from mlflow.models.resources import (
    DatabricksVectorSearchIndex
)

# Set Model Registry URI to Unity Catalog
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(f"/Shared/rag_app")

model_name = f"{catalog_name}.{schema_name}.rag_app"
input_example = {"messages": [ {"role": "user", "content": "What is Retrieval-augmented Generation?"}]}

# Register the assembled RAG model in Model Registry with Unity Catalog
signature = infer_signature(input_example, answer)
model_info = mlflow.langchain.log_model(
    lc_model=chain,
    name="chain",
    input_example=input_example,
    signature=signature,
    pip_requirements=[
        "langchain==" + langchain.__version__,
        "databricks-vectorsearch==" + databricks.vector_search.__version__,
        "databricks-langchain",
        "mlflow==" + mlflow.__version__
    ],
    resources=[
        DatabricksVectorSearchIndex(index_name=vs_index_fullname)
    ]
)

mlflow.register_model(model_info.model_uri, model_name)

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>